In [1]:
import pandas as pd

df = pd.read_csv("../Planilhas/imoveis_unificado.csv", sep=";")
#primeira linha do csv veio com o nome das colunas duplicados, então vamos ignorar a primeira linha por enquanto
df = df.iloc[1:].reset_index(drop=True)


print(df.shape)
df.info()
df.nunique(dropna=False)

/tmp/ipykernel_175547/1842045325.py:3: DtypeWarning: Columns (1,7,18,19,20,21,22) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../Planilhas/imoveis_unificado.csv", sep=";")


(44233, 23)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44233 entries, 0 to 44232
Data columns (total 23 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   logradouro          44233 non-null  object
 1   numero              44233 non-null  object
 2   complemento         40084 non-null  object
 3   valor_avaliacao     44233 non-null  object
 4   bairro              44233 non-null  object
 5   cidade              44233 non-null  object
 6   uf                  44233 non-null  object
 7   ano_construcao      44233 non-null  object
 8   area_terreno        44233 non-null  object
 9   area_construida     44233 non-null  object
 10  fracao_ideal        44233 non-null  object
 11  padrao_acabamento   44233 non-null  object
 12  tipo_construcao     44233 non-null  object
 13  tipo_ocupacao       44233 non-null  object
 14  data_transacao      44233 non-null  object
 15  estado_conservacao  44233 non-null  object
 16  tipo_imove

logradouro             2238
numero                 2960
complemento           29206
valor_avaliacao       14793
bairro                   96
cidade                    2
uf                        2
ano_construcao          177
area_terreno           5478
area_construida        9380
fracao_ideal           4653
padrao_acabamento         4
tipo_construcao          17
tipo_ocupacao             4
data_transacao          902
estado_conservacao        4
tipo_imovel              20
sfh                    7917
cod_logradouro         2996
latitude               8464
longitude              8335
ano                       5
distrito                  8
dtype: int64

# Tratamento da Coluna Complemento
Obs: coluna com maior quantidade de valores unicos

Observações iniciais dos dados

In [2]:
df['complemento'].info()
print(f"Total de nulos: {df['complemento'].isna().sum()}")
print(f"Total de linhas: {len(df)}")

<class 'pandas.core.series.Series'>
RangeIndex: 44233 entries, 0 to 44232
Series name: complemento
Non-Null Count  Dtype 
--------------  ----- 
40084 non-null  object
dtypes: object(1)
memory usage: 345.7+ KB
Total de nulos: 4149
Total de linhas: 44233


In [3]:
print(df['complemento'].value_counts(dropna=False).head(20))


complemento
NaN                  4149
apto 0101              53
apto 2                 49
apto 0102              46
apto 0201              45
apto 1                 43
apto 0202              32
apto 0102 bloco a      28
apto 302 bloco a       27
apto 302 bloco b       27
apto 203 bloco a       27
apto 102 bloco a       26
apto 4                 26
apto 401 bloco a       25
apto 502 bloco a       24
apto 303 bloco a       24
apto 402 bloco a       24
apto 601 bloco a       24
apto 501 bloco b       24
apto 804 bloco a       24
Name: count, dtype: int64


In [4]:
df['complemento'].head(20)

0                                NaN
1                                NaN
2                             apto 1
3                             apto 1
4                             apto 2
5                             apto 3
6                             apto 4
7                          apto 0005
8                                NaN
9                                NaN
10       apto 203 edf sainte juliana
11       apto 201 edf sainte juliana
12      apto 1501 edf sainte juliana
13                               NaN
14    apto 902 edf essenza rosarinho
15       apto 2101 edf. jardim monet
16      apto 1401 edf praia de ceres
17           apto 201 edf maria rosa
18           apto 404 edf maria rosa
19          apto 1304 edf maria rosa
Name: complemento, dtype: object

Aqui já foi percebido uma falta de padronização nos complementos, alguns com numeração (0101) e (101), outros até com nome do edifício

In [5]:
# Ver todos os padrões com "edf" para entender as variações
mask = df['complemento'].str.contains('edf', na=False)
print(df[mask]['complemento'].value_counts().head(30))

complemento
apto 401 edf ocean breeze                       9
apto 0601 edf maria gabriela                    8
sala 2207 edf. torreão executive plaza          7
apto 0802 edf saint paul                        7
apto 0701 edf n sra do amparo                   5
apto 0501 edf hockenheim                        5
apto 204 edf jardins madalena                   5
apto 1101 edf rita de cassia - bloco a          5
apto 1601 edf vivres village                    5
apto 1402 edf rita de cassia - bloco a          5
apto 1102 edf maria eugenia - bloco c           5
apto 1507 edf wimbledon boa viagem              5
apto 305 edf portal da varzea                   5
sala 0609 edf unicenter emp                     5
apto 0101 edf bom pr                            5
apto 2203 edf praça dos mognos                  5
apto 102 edf flor de macambira                  5
sala 0009 edf canarias                          5
apto 1009 edf. tolive one                       5
apto 0202 edf baía helsink            

In [6]:
#Buscar variações com apt e apto
mask = df['complemento'].str.contains('apt\.', na=False, case=False)
print(df[mask]['complemento'].value_counts().head(20))
print(f"\nTotal: {mask.sum()}")

Series([], Name: count, dtype: int64)

Total: 0


## Tratamento da coluna `complemento`

A coluna apresenta dois tipos de informação misturados em texto livre:
- Unidade do imóvel: `apto 203`, `sala 0609`
- Nome do edifício: `edf sainte juliana`, `edf. tolive one`

### Decisões de tratamento:
1. Padronizar texto para maiúsculo e remover espaços extras
2. Separar em duas novas colunas: `complemento_numero` e `nome_edificio`
3. Manter nulos como `NaN` — ausência de complemento é válida
4. Remover a coluna original após a separação

In [7]:
#Deixar todo o texto em CapsLk
df['complemento'] = df['complemento'].str.strip().str.upper()

#Extrair número da unidade (apto/sala + número)
df['complemento_numero'] = df['complemento'].str.extract(
    r'^((?:APTO|SALA)\s+[\w]+)',
    expand=False
)
#Extrair nome do edifício (tudo após EDF. ou EDF)
df['nome_edificio'] = df['complemento'].str.extract(
    r'EDF\.?\s+(.+)',
    expand=False
).str.strip()

#Remover a coluna original
df = df.drop(columns=['complemento'])

#Verificando se deu certo
print(df[['complemento_numero', 'nome_edificio']].head(20))
print(f"\nNulos em complemento_numero: {df['complemento_numero'].isna().sum()}")
print(f"Nulos em nome_edificio: {df['nome_edificio'].isna().sum()}")

   complemento_numero      nome_edificio
0                 NaN                NaN
1                 NaN                NaN
2              APTO 1                NaN
3              APTO 1                NaN
4              APTO 2                NaN
5              APTO 3                NaN
6              APTO 4                NaN
7           APTO 0005                NaN
8                 NaN                NaN
9                 NaN                NaN
10           APTO 203     SAINTE JULIANA
11           APTO 201     SAINTE JULIANA
12          APTO 1501     SAINTE JULIANA
13                NaN                NaN
14           APTO 902  ESSENZA ROSARINHO
15          APTO 2101       JARDIM MONET
16          APTO 1401     PRAIA DE CERES
17           APTO 201         MARIA ROSA
18           APTO 404         MARIA ROSA
19          APTO 1304         MARIA ROSA

Nulos em complemento_numero: 5342
Nulos em nome_edificio: 17987


## Conclusão da separação da coluna `complemento`

A coluna `complemento` original apresentava duas informações distintas
em um único campo de texto livre: a unidade do imóvel (ex: `apto 203`)
e o nome do edifício (ex: `edf sainte juliana`), o que tornava a coluna
desorganizada e dificultava qualquer análise isolada dessas informações.

Por isso, optou-se por separar em duas colunas:
- `complemento_numero`: identifica a unidade do imóvel
- `nome_edificio`: identifica o condomínio ou edifício

Descartar a informação do edifício não seria adequado, uma vez que
aproximadamente **26.246 linhas** possuem esse campo preenchido,
representando uma parcela significativa do dataset e sendo essencial
para análises de imóveis específicos.

#### Organizar posição das colunas após adaptação

In [8]:
colunas = [
    'logradouro', 'numero', 'complemento_numero', 'nome_edificio',
    'valor_avaliacao', 'bairro', 'cidade', 'uf', 'ano_construcao',
    'area_terreno', 'area_construida', 'fracao_ideal',
    'padrao_acabamento', 'tipo_construcao', 'tipo_ocupacao',
    'data_transacao', 'estado_conservacao', 'tipo_imovel',
    'sfh', 'cod_logradouro', 'latitude', 'longitude', 'ano', 'distrito'
]

df = df[colunas]
print(df.columns.tolist())

['logradouro', 'numero', 'complemento_numero', 'nome_edificio', 'valor_avaliacao', 'bairro', 'cidade', 'uf', 'ano_construcao', 'area_terreno', 'area_construida', 'fracao_ideal', 'padrao_acabamento', 'tipo_construcao', 'tipo_ocupacao', 'data_transacao', 'estado_conservacao', 'tipo_imovel', 'sfh', 'cod_logradouro', 'latitude', 'longitude', 'ano', 'distrito']


# Tratamento da coluna 'tipo_imovel'

observação inicial dos dados

In [9]:
df['tipo_imovel'].info()


<class 'pandas.core.series.Series'>
RangeIndex: 44233 entries, 0 to 44232
Series name: tipo_imovel
Non-Null Count  Dtype 
--------------  ----- 
44233 non-null  object
dtypes: object(1)
memory usage: 345.7+ KB


In [10]:
print(df['tipo_imovel'].value_counts(dropna=False))
print(f"\nTotal de nulos: {df['tipo_imovel'].isna().sum()}")

tipo_imovel
Apartamento                    35697
Casa                            3574
Sala                            3096
Loja                             790
Edificação Especial              319
Galpão                           269
Centro Comercial/Serviços        158
Hotel                             73
Garagem Comercial                 59
Instituição Financeira            49
Garagem Residencial               33
Industria                         31
Instituição Educacional           20
Hospital                          20
Mocambo                           15
Posto de Abastecimento            15
Templo religioso                   5
Galpão Fechado                     4
Terreno em cond residencial        4
tipo_imovel                        2
Name: count, dtype: int64

Total de nulos: 0


Como essa coluna está bem padronizada, apenas iremos confirmar a quantidade de categorias que temos

In [11]:
print(f"Categorias únicas: {df['tipo_imovel'].nunique()}")

Categorias únicas: 20


## Tratamento da coluna `tipo_imovel`

A coluna `tipo_imovel` classifica o tipo de imóvel envolvido em cada
transação. Após análise, a coluna apresentou um estado consideravelmente
limpo, com as seguintes características:

- **0 valores nulos**
- **19 categorias bem definidas**, sem variações de escrita ou abreviações
  (ex: não havia `"Apto"` vs `"Apartamento"`)


### Decisão de tratamento:
Dado o bom estado da coluna, foi decidido que não é necessário ser feito nenhum tratamento,
a coluna está pronta para uso.

# Tratamento da coluna 'data_transacao'

In [12]:
print(df['data_transacao'].dtype)
print(df['data_transacao'].value_counts(dropna=False).head(10))
print(f"\nTotal de nulos: {df['data_transacao'].isna().sum()}")
print(f"Amostra: {df['data_transacao'].sample(5).tolist()}")

object
data_transacao
2025-11-14    253
2025-07-01    215
2024-05-21    213
2025-01-08    191
2024-05-20    160
2024-09-09    155
2025-09-16    147
2025-05-19    147
2025-06-03    144
2025-07-18    144
Name: count, dtype: int64

Total de nulos: 0
Amostra: ['2024-09-30', '2025-11-06', '2024-12-03', '2025-09-25', '2025-03-07']


Percebemos que o tipo do dado está como objeto, então vamos transforma-lo em data

In [13]:
#Nessa seção, somos forçados a remover o valor duplicado do início, pois, isso causa erro na mudança de tipo
df = df[df['data_transacao'] != 'data_transacao'].reset_index(drop=True)

df['data_transacao'] = pd.to_datetime(df['data_transacao'], format='%Y-%m-%d')

print(df['data_transacao'].dtype)


datetime64[ns]


In [14]:
df['ano'] = df['ano'].astype(int)
df['ano_transacao'] = df['data_transacao'].dt.year


# Agora refaz o teste
divergencias = df[df['ano'] != df['ano_transacao']]
print(f"Total de divergências: {len(divergencias)}")
print(divergencias.groupby(['ano', 'ano_transacao']).size())
df = df.drop(columns=['ano_transacao'])


Total de divergências: 0
Series([], dtype: int64)


## Tratamento da coluna `data_transacao`

A coluna `data_transacao` registra a data em que a transação imobiliária
foi realizada. Após análise, a coluna apresentou as seguintes características:

- **0 valores nulos**
- **Formato consistente: `YYYY-MM-DD`** em todos os registros
- **dtype `object`** — a coluna estava armazenada como texto, sendo necessária
  a conversão para o tipo `datetime`

### Decisões de tratamento:
1. Conversão da coluna para o tipo `datetime` com formato `%Y-%m-%d`

Optou-se por **manter a coluna intacta após a conversão**, sem extrair
colunas auxiliares de mês, trimestre ou dia. A coluna `data_transacao`
como `datetime` já permite essas extrações diretamente no momento da
análise, evitando colunas desnecessárias no dataset.